# Лабораторная 2: декодерный трансформер на `Tiny Shakespeare`

Цель работы: перенести механизм авторегрессионного предсказания следующего токена
с детерминированной синтетики на реальный текстовый корпус.


## Маршрут выполнения

Строгий порядок шагов:
1. `TODO 1`: загрузка корпуса и построение словаря.
2. `TODO 2`: окна фиксированной длины и индексное разбиение.
3. `TODO 3`: декодерный блок с причинной маской.
4. `TODO 4`: обучение, метрики и сравнение с частотным ориентиром.
5. `TODO 5`: детерминированная генерация по фиксированным подсказкам.
6. `TODO 6`: диагностика внимания без доступа в будущее.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

RUNTIME_MODE = os.environ.get("COURSE_RUNTIME_MODE", "auto")
COURSE_REPO_HTTPS_URL = os.environ.get(
    "COURSE_REPO_HTTPS_URL",
    "https://github.com/<org>/<repo>.git",
)
NOTEBOOK_REQUIREMENTS = "themes/04-Autoregression/lab/requirements.txt"


def _detect_notebook_platform():
    """Определяет тип среды выполнения текущей тетради.

    Args:
      Нет.

    Returns:
      Строка из множества `{'local', 'colab', 'kaggle'}`.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    return "local"


def _looks_like_repo_root(path: Path) -> bool:
    """Проверяет, похож ли путь на корень учебного репозитория.

    Args:
      path: Проверяемый путь.

    Returns:
      `True`, если обнаружены ключевые признаки корня репозитория.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return (
        path.is_dir()
        and (path / "themes").is_dir()
        and (path / "course_runtime.py").is_file()
    )


def _canonical_cloud_repo_root(platform: str) -> Path:
    """Возвращает стандартный путь клонирования для облачной платформы.

    Args:
      platform: Имя платформы (`'colab'` или `'kaggle'`).

    Returns:
      Абсолютный путь каталога репозитория.

    Raises:
      ValueError: Если передано неподдерживаемое имя платформы.
    """
    if platform == "colab":
        return Path("/content/students-AI_math_essentials")
    if platform == "kaggle":
        return Path("/kaggle/working/students-AI_math_essentials")
    raise ValueError(f"Unexpected cloud platform: {platform}")


def _is_placeholder_repo_url(repo_url: str) -> bool:
    """Проверяет, остался ли в настройке шаблонный URL репозитория.

    Args:
      repo_url: Проверяемый URL репозитория.

    Returns:
      `True`, если URL имеет вид шаблона-заглушки.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return repo_url.strip() == "https://github.com/<org>/<repo>.git"


def _find_repo_root_from_cwd():
    """Ищет корень курса, поднимаясь от текущего каталога вверх.

    Args:
      Нет.

    Returns:
      Объект `Path` корня репозитория или `None`, если путь не найден.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if _looks_like_repo_root(candidate):
            return candidate
    return None


def _ensure_course_runtime_importable(runtime_mode: str, repo_url: str) -> None:
    """Обеспечивает доступность модуля `course_runtime` для текущей среды.

    Args:
      runtime_mode: Режим запуска тетради.
      repo_url: URL репозитория курса для облачной автозагрузки.

    Returns:
      `None`.

    Raises:
      ModuleNotFoundError: Если локальный запуск выполнен вне корректного корня репозитория.
      RuntimeError: Если в облаке отсутствует валидный URL репозитория или каталог повреждён.
      subprocess.CalledProcessError: Если команда `git clone` завершается с ошибкой.
    """
    if importlib.util.find_spec("course_runtime") is not None:
        return

    local_repo_root = _find_repo_root_from_cwd()
    if local_repo_root is not None:
        if str(local_repo_root) not in sys.path:
            sys.path.insert(0, str(local_repo_root))
        return

    platform = _detect_notebook_platform()
    if platform == "local":
        raise ModuleNotFoundError(
            "Не удалось импортировать course_runtime.py. Для локального запуска "
            "открывайте репозиторий через `.venv/bin/jupyter notebook` из корня проекта."
        )

    repo_root = _canonical_cloud_repo_root(platform)
    if not _looks_like_repo_root(repo_root):
        if _is_placeholder_repo_url(repo_url):
            raise RuntimeError(
                "Для cloud auto-bootstrap нужен публичный HTTPS URL курса. "
                "Замените COURSE_REPO_HTTPS_URL на реальный адрес репозитория."
            )
        repo_root.parent.mkdir(parents=True, exist_ok=True)
        if repo_root.exists() and any(repo_root.iterdir()):
            raise RuntimeError(
                f"Каталог {repo_root} уже существует, но не выглядит как корень курса. "
                "Очистите runtime или используйте новый notebook session."
            )
        print(f"Bootstrapping course repository into {repo_root} ...")
        subprocess.run(["git", "clone", repo_url, str(repo_root)], check=True)

    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))


_ensure_course_runtime_importable(RUNTIME_MODE, COURSE_REPO_HTTPS_URL)

from course_runtime import setup_notebook_runtime

runtime_info = setup_notebook_runtime(
    runtime_mode=RUNTIME_MODE,
    course_repo_https_url=COURSE_REPO_HTTPS_URL,
    notebook_requirements=NOTEBOOK_REQUIREMENTS,
)
runtime_info.as_dict()


## Константы и профиль выполнения

В `ЛР02` обязательным является профиль `CPU-friendly`.
Профиль `GPU-friendly` выполняется дополнительно при наличии графического процессора.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path

SEED = 17
PROFILE = 'cpu'  # Допустимые значения: 'cpu', 'gpu'

PROFILE_CONFIG = {
    'cpu': {
        'chars': 250_000,
        'context': 64,
        'stride': 3,
        'batch_size': 64,
        'embed_dim': 96,
        'num_heads': 4,
        'ff_dim': 192,
        'epochs': 3,
        'learning_rate': 2e-3,
        'gen_match_ratio': 0.20,
        'gen_threshold': 5,
        'gen_mean_threshold': 0.11,
    },
    'gpu': {
        'chars': 900_000,
        'context': 128,
        'stride': 1,
        'batch_size': 128,
        'embed_dim': 192,
        'num_heads': 6,
        'ff_dim': 384,
        'epochs': 4,
        'learning_rate': 1e-3,
        'gen_match_ratio': 0.25,
        'gen_threshold': 7,
        'gen_mean_threshold': 0.13,
    },
}

if PROFILE not in PROFILE_CONFIG:
    raise ValueError("PROFILE должен быть равен 'cpu' или 'gpu'.")

cfg = PROFILE_CONFIG[PROFILE]
PAD_ID = 0
CHECK_GEN_STEPS = 12
PROMPT_COUNT = 20

plt.style.use('default')
keras.utils.set_random_seed(SEED)


## Математический ориентир

Мы оптимизируем токенную перекрёстную энтропию (cross-entropy), которая эквивалентна
среднему отрицательному лог-правдоподобию для next-token задачи.

Перплексия (perplexity) вычисляется как:

$$
\mathrm{PPL} = e^{\mathrm{loss}}
$$

Дополнительно в этой работе используется частотный базовый ориентир:
если модель не превосходит его по перплексии, перенос на реальный корпус считается недостаточным.


## TODO 1: загрузка корпуса и построение словаря


In [ ]:
def load_tiny_shakespeare(profile_cfg):
    """Загружает корпус и строит детерминированное символьное кодирование.

    Args:
      profile_cfg: Словарь параметров выбранного профиля.

    Returns:
      Кортеж `(text, encoded_ids, vocab, char_to_id, id_to_char)`.

    Raises:
      ValueError: Если выбранный профиль задаёт слишком короткий фрагмент текста.
    """
    # TODO 1.1: Загрузите корпус Tiny Shakespeare через tf.keras.utils.get_file.
    # TODO 1.2: Возьмите детерминированный срез длины profile_cfg['chars'].
    # TODO 1.3: Постройте словарь ['<PAD>', ...] и кодирование в int32.
    raise NotImplementedError('TODO 1: реализуйте load_tiny_shakespeare')


# TODO 1.4: Вызовите load_tiny_shakespeare(cfg) и сохраните результаты в
# text, encoded_ids, vocab, char_to_id, id_to_char.
raise NotImplementedError('TODO 1: выполните загрузку корпуса')


In [ ]:
# Мини-проверка после TODO 1
assert len(text) == cfg['chars'], 'Длина среза текста не совпадает с профилем.'
assert encoded_ids.dtype == np.int32, 'Кодирование должно быть int32.'
assert vocab[PAD_ID] == '<PAD>', 'Нулевой идентификатор должен быть зарезервирован под PAD.'

text_b, ids_b, _, _, _ = load_tiny_shakespeare(cfg)
assert np.array_equal(encoded_ids[:1000], ids_b[:1000]), 'Кодирование должно быть воспроизводимым.'

print('Профиль:', PROFILE)
print('Длина текста:', len(text))
print('Размер словаря:', len(vocab))


## TODO 2: окна фиксированной длины и разбиение


In [ ]:
def build_windows(encoded_stream, context_len, stride):
    """Строит обучающие окна фиксированной длины.

    Args:
      encoded_stream: Одномерный массив кодов символов.
      context_len: Длина входного контекста.
      stride: Шаг между соседними окнами.

    Returns:
      Кортеж `(X, Y, starts)`.

    Raises:
      ValueError: Если поток слишком короткий для построения хотя бы одного окна.
    """
    # TODO 2.1: Постройте X и Y со сдвигом на один шаг.
    # TODO 2.2: Верните также массив стартовых индексов starts.
    raise NotImplementedError('TODO 2: реализуйте build_windows')


# TODO 2.3: Выполните индексное разбиение X_all, y_all, starts_all на
# train/val/test без случайного перемешивания.
raise NotImplementedError('TODO 2: реализуйте детерминированное разбиение')


In [ ]:
# Мини-проверка после TODO 2
assert X_train.shape[1] == cfg['context']
assert y_train.shape == X_train.shape
assert starts_train.ndim == 1

# Проверка сдвига внутри окон.
assert np.array_equal(X_train[0, 1:], y_train[0, :-1]), 'Сдвиг X/Y нарушен.'

print('Окон train/val/test:', len(X_train), len(X_val), len(X_test))


## TODO 3: декодерный блок с причинной маской


In [ ]:
def build_causal_mask(seq_len):
    """Строит нижнетреугольную причинную маску.

    Args:
      seq_len: Длина последовательности.

    Returns:
      Булев тензор формы `(seq_len, seq_len)`.

    Raises:
      tf.errors.InvalidArgumentError: Если `seq_len` не является положительным.
    """
    # TODO 3.1: Реализуйте построение нижнетреугольной маски. Используйте тензорную
    # проверку `tf.debugging.assert_positive`, чтобы код работал и в графовом режиме.
    raise NotImplementedError('TODO 3: реализуйте build_causal_mask')


class TokenAndPositionEmbedding(layers.Layer):
    """Складывает токенные и позиционные векторы.

    Args:
      maxlen: Максимальная длина контекста.
      vocab_size: Размер словаря.
      embed_dim: Размер скрытого представления.
      **kwargs: Дополнительные аргументы базового слоя.

    Returns:
      Экземпляр слоя встраивания.

    Raises:
      ValueError: Если `embed_dim` меньше 1.
    """

    def __init__(self, maxlen, vocab_size, embed_dim, **kwargs):
        """Инициализирует слой токенного и позиционного встраивания.

        Args:
          maxlen: Максимальная длина контекста.
          vocab_size: Размер словаря токенов.
          embed_dim: Размерность векторного представления.
          **kwargs: Дополнительные аргументы базового слоя Keras.

        Returns:
          None.

        Raises:
          ValueError: Если `embed_dim` меньше 1.
        """
        super().__init__(**kwargs)
        if embed_dim < 1:
            raise ValueError('embed_dim должен быть положительным.')
        self.token_embedding = layers.Embedding(vocab_size, embed_dim, mask_zero=True)
        self.position_embedding = layers.Embedding(maxlen, embed_dim)

    def call(self, inputs):
        """Суммирует токенные и позиционные векторы.

        Args:
          inputs: Матрица идентификаторов формы `(batch, time)`.

        Returns:
          Тензор формы `(batch, time, embed_dim)`.

        Raises:
          NotImplementedError: Пока шаг `TODO 3` не реализован.
        """
        # TODO 3.2: Реализуйте сложение token embedding и position embedding.
        raise NotImplementedError('TODO 3: реализуйте TokenAndPositionEmbedding.call')

    def compute_mask(self, inputs, mask=None):
        """Пробрасывает маску непустых токенов.

        Args:
          inputs: Матрица токенов.
          mask: Входная маска.

        Returns:
          Булева маска формы `(batch, time)`.

        Raises:
          RuntimeError: Не выбрасывается в штатном режиме.
        """
        return self.token_embedding.compute_mask(inputs)


class CausalDecoderBlock(layers.Layer):
    """Минимальный декодерный блок с причинной маской.

    Args:
      embed_dim: Размер скрытого представления.
      num_heads: Число голов внимания.
      ff_dim: Размер скрытого слоя позиционно-независимой сети.
      rate: Доля прореживания.
      **kwargs: Дополнительные аргументы базового слоя.

    Returns:
      Экземпляр декодерного блока.

    Raises:
      ValueError: Если `embed_dim` не делится на `num_heads`.
    """

    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        """Создаёт внутренние слои декодерного блока.

        Args:
          embed_dim: Размерность входных признаков.
          num_heads: Число голов внимания.
          ff_dim: Размер скрытого слоя позициионно-независимой сети.
          rate: Доля выключаемых нейронов в прореживании.
          **kwargs: Дополнительные аргументы базового слоя Keras.

        Returns:
          None.

        Raises:
          ValueError: Если `embed_dim` не делится на `num_heads`.
        """
        super().__init__(**kwargs)
        if embed_dim % num_heads != 0:
            raise ValueError('embed_dim должен делиться на num_heads без остатка.')
        self.self_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=rate,
        )
        self.ffn = keras.Sequential(
            [
                layers.Dense(ff_dim, activation='relu'),
                layers.Dense(embed_dim),
            ]
        )
        self.norm_1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm_2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout_1 = layers.Dropout(rate)
        self.dropout_2 = layers.Dropout(rate)

    def call(self, inputs, padding_mask=None, training=None, return_attention_scores=False):
        """Прогоняет вход через маскированное самовнимание и FFN.

        Args:
          inputs: Тензор формы `(batch, time, embed_dim)`.
          padding_mask: Булева маска формы `(batch, time)`.
          training: Признак режима обучения.
          return_attention_scores: Нужно ли вернуть веса внимания.

        Returns:
          Либо выходной тензор, либо кортеж `(выход, attention_scores)`.

        Raises:
          NotImplementedError: Пока шаг `TODO 3` не реализован.
        """
        # TODO 3.3: Реализуйте causal mask + padding mask и прямой проход блока.
        raise NotImplementedError('TODO 3: реализуйте CausalDecoderBlock.call')


def masked_sparse_crossentropy(y_true, y_pred):
    """Считает перекрёстную энтропию по непустым позициям.

    Args:
      y_true: Истинные токены.
      y_pred: Предсказанные распределения.

    Returns:
      Среднее значение функции потерь.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    per_token = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    return tf.reduce_sum(per_token * mask) / tf.reduce_sum(mask)


def masked_token_accuracy(y_true, y_pred):
    """Считает токенную точность по непустым позициям.

    Args:
      y_true: Истинные токены.
      y_pred: Предсказанные распределения.

    Returns:
      Доля верных токенов.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    pred = tf.argmax(y_pred, axis=-1, output_type=y_true.dtype)
    correct = tf.cast(tf.equal(y_true, pred), tf.float32)
    mask = tf.cast(tf.not_equal(y_true, PAD_ID), tf.float32)
    return tf.reduce_sum(correct * mask) / tf.reduce_sum(mask)


def perplexity_from_loss(loss_value):
    """Преобразует значение функции потерь в перплексию.

    Args:
      loss_value: Средняя перекрёстная энтропия.

    Returns:
      Значение перплексии.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return float(np.exp(loss_value))


def frequency_baseline_perplexity(y_train_data, y_test_data, vocab_size, pad_id=PAD_ID):
    """Считает частотный базовый ориентир перплексии.

    Args:
      y_train_data: Целевые токены обучающей выборки.
      y_test_data: Целевые токены тестовой выборки.
      vocab_size: Размер словаря.
      pad_id: Идентификатор дополнения.

    Returns:
      Значение базовой перплексии.

    Raises:
      ValueError: Если в данных нет полезных токенов.
    """
    train_tokens = y_train_data[y_train_data != pad_id].reshape(-1)
    test_tokens = y_test_data[y_test_data != pad_id].reshape(-1)
    if len(train_tokens) == 0 or len(test_tokens) == 0:
        raise ValueError('Для базового ориентира нужны непустые токены.')

    counts = np.bincount(train_tokens, minlength=vocab_size).astype(np.float64)
    probs = counts / counts.sum()
    probs = np.maximum(probs, 1e-12)

    nll = -np.mean(np.log(probs[test_tokens]))
    return float(np.exp(nll))


## TODO 4: сборка и обучение модели


In [ ]:
# TODO 4.1: Соберите модель decoder-only с TokenAndPositionEmbedding и CausalDecoderBlock.
# TODO 4.2: Скомпилируйте модель с masked_sparse_crossentropy и masked_token_accuracy.
# TODO 4.3: Обучите модель на train_ds и val_ds в детерминированном порядке.
# TODO 4.4: Посчитайте test_loss, test_accuracy, test_perplexity и baseline_perplexity.
raise NotImplementedError('TODO 4: соберите и обучите модель')


In [ ]:
# Графики и контроль после TODO 4
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()

plt.subplot(1, 2, 2)
train_ppl = [perplexity_from_loss(v) for v in history.history['loss']]
val_ppl = [perplexity_from_loss(v) for v in history.history['val_loss']]
plt.plot(train_ppl, label='train_ppl')
plt.plot(val_ppl, label='val_ppl')
plt.xlabel('epoch')
plt.ylabel('perplexity')
plt.legend()
plt.tight_layout()

print(f"test_loss            : {test_loss:.4f}")
print(f"test_token_accuracy  : {test_token_accuracy:.4f}")
print(f"test_perplexity      : {test_perplexity:.4f}")
print(f"baseline_perplexity  : {baseline_perplexity:.4f}")

assert test_perplexity < baseline_perplexity, 'Модель должна превосходить частотный ориентир.'


## TODO 5: детерминированная генерация по фиксированным подсказкам


In [ ]:
def ids_to_text(token_ids, id_to_char_map):
    """Преобразует идентификаторы в строку.

    Args:
      token_ids: Последовательность идентификаторов.
      id_to_char_map: Отображение id -> символ.

    Returns:
      Строка символов.

    Raises:
      RuntimeError: Не выбрасывается в штатном режиме.
    """
    return ''.join(id_to_char_map.get(int(token), '') for token in token_ids if int(token) != PAD_ID)


def generate_greedy(model, prompt_ids, steps, context_len):
    """Генерирует продолжение в режиме argmax.

    Args:
      model: Обученная модель.
      prompt_ids: Начальная последовательность идентификаторов.
      steps: Число новых токенов.
      context_len: Длина модельного контекста.

    Returns:
      Список токенов продолжения длины не больше `steps`.

    Raises:
      ValueError: Если подсказка пуста.
    """
    # TODO 5.1: Реализуйте генерацию argmax без случайной выборки.
    raise NotImplementedError('TODO 5: реализуйте generate_greedy')


def build_prompt_targets(encoded_stream, start_indices, context_len, continuation_len, n_prompts):
    """Готовит фиксированные подсказки и эталонные продолжения из тестового фрагмента.

    Args:
      encoded_stream: Полный поток кодов корпуса.
      start_indices: Стартовые индексы окон тестовой части.
      context_len: Длина контекста модели.
      continuation_len: Длина целевого продолжения.
      n_prompts: Количество подсказок.

    Returns:
      Список пар `(prompt_ids, target_ids)`.

    Raises:
      ValueError: Если тестовых окон недостаточно.
    """
    if len(start_indices) < n_prompts:
        raise ValueError('Недостаточно тестовых окон для фиксированного набора подсказок.')

    selected = np.linspace(0, len(start_indices) - 1, n_prompts, dtype=int)
    pairs = []
    for idx in selected:
        start = int(start_indices[idx])
        prompt = encoded_stream[start:start + context_len]
        target = encoded_stream[start + context_len:start + context_len + continuation_len]
        if len(prompt) == context_len and len(target) == continuation_len:
            pairs.append((prompt.tolist(), target.tolist()))
    return pairs


# TODO 5.2: Постройте fixed_pairs через build_prompt_targets(...), затем
# посчитайте `success_count` и `mean_match_ratio` по фиксированным подсказкам.
# Подсказка: успех по одной подсказке фиксируйте, если
# `match_ratio >= cfg['gen_match_ratio']`.
# Итоговые пороги: `success_count >= cfg['gen_threshold']` и
# `mean_match_ratio >= cfg['gen_mean_threshold']`.
raise NotImplementedError('TODO 5: выполните проверку генерации')


## TODO 6: диагностика внимания


In [ ]:
# TODO 6.1: Постройте trace-модель, которая возвращает attention_scores из decoder_layer.
# TODO 6.2: Усредните веса внимания по головам и вычислите отношение массы в будущем.
# TODO 6.3: Проверьте, что отношение массы в будущем меньше 1e-4.
raise NotImplementedError('TODO 6: выполните диагностику внимания')


## Критерии завершения `ЛР02`

1. Все `TODO` закрыты.
2. Профиль `CPU-friendly` пройден полностью.
3. `test_perplexity < baseline_perplexity`.
4. Порог генерации достигнут: `success_count >= cfg['gen_threshold']`.
5. Диагностика внимания подтверждает отсутствие доступа в будущее.
6. Для `GPU-friendly` показано улучшение относительно `CPU-friendly` по перплексии.
